# EDA — Presidential Speeches Corpus

Exploratory analysis of `data/raw/speeches_raw.csv` produced by `signal/scraping/scraper.py`.

**Goals**
- Assess corpus coverage and data quality
- Understand speech volume and length by president and over time
- Identify months with no speeches (relevant for BVAR interpolation decisions)
- Flag anomalies before the preprocessing and scoring stages

In [ ]:
%matplotlib inline

import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
PRES_COLORS = {
    "Macri":   "#2196F3",
    "AF":      "#4CAF50",
    "Milei":   "#FF5722",
    "CFK":     "#9C27B0",
    "Unknown": "#9E9E9E",
}
PRES_ORDER = ["Macri", "AF", "Milei", "CFK", "Unknown"]
plt.rcParams["figure.dpi"] = 120

## 1. Load & basic checks

In [ ]:
df = pd.read_csv("../data/raw/speeches_raw.csv", parse_dates=["date"])

print(f"Speeches : {len(df):,}")
print(f"Date range : {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Columns  : {df.columns.tolist()}")
df.head(3)

In [ ]:
print("Nulls per column:")
print(df.isnull().sum().to_string())

In [ ]:
anomalies = df[df["president"].isin(["Unknown", "CFK"]) | df["date"].isnull()]
print(f"{len(anomalies)} anomalous rows:")
anomalies[["speech_id", "date", "president", "n_words", "url"]]

## 2. Corpus overview by president

In [ ]:
summary = (
    df.groupby("president", observed=True)
    .agg(
        n_speeches=("speech_id", "count"),
        total_words=("n_words", "sum"),
        mean_words=("n_words", "mean"),
        median_words=("n_words", "median"),
        first_speech=("date", "min"),
        last_speech=("date", "max"),
    )
    .loc[[p for p in PRES_ORDER if p in df["president"].unique()]]
    .reset_index()
)
summary["mean_words"]   = summary["mean_words"].round(0).astype(int)
summary["median_words"] = summary["median_words"].round(0).astype(int)
summary["total_words"]  = summary["total_words"].astype(int)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
core = summary[summary["president"].isin(["Macri", "AF", "Milei"])]

axes[0].bar(core["president"], core["n_speeches"],
            color=[PRES_COLORS[p] for p in core["president"]])
axes[0].set_title("Speeches in corpus")
axes[0].set_ylabel("Count")
for bar, val in zip(axes[0].patches, core["n_speeches"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha="center", va="bottom", fontsize=10)

axes[1].bar(core["president"], core["total_words"] / 1e6,
            color=[PRES_COLORS[p] for p in core["president"]])
axes[1].set_title("Total words (millions)")
axes[1].set_ylabel("Million words")

plt.tight_layout()
plt.savefig("../outputs/figures/01_corpus_overview.png", bbox_inches="tight")
plt.show()

## 3. Speech volume over time

In [ ]:
monthly = (
    df[df["date"].notna()]
    .assign(ym=lambda d: d["date"].dt.to_period("M"))
    .groupby(["ym", "president"], observed=True)
    .size()
    .reset_index(name="n_speeches")
)
monthly["ym_dt"] = monthly["ym"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 4))
for pres in ["Macri", "AF", "Milei"]:
    sub = monthly[monthly["president"] == pres]
    ax.bar(sub["ym_dt"], sub["n_speeches"],
           color=PRES_COLORS[pres], label=pres, width=25, alpha=0.85)

for date, label in [("2015-12-10", "Macri"), ("2019-12-10", "AF"), ("2023-12-10", "Milei")]:
    ax.axvline(pd.Timestamp(date), color="black", linewidth=1, linestyle="--", alpha=0.5)
    ax.text(pd.Timestamp(date), ax.get_ylim()[1]*0.9, label,
            fontsize=8, ha="left", rotation=90, alpha=0.6)

ax.set_title("Monthly speech volume by president (2015-2026)")
ax.set_ylabel("Speeches per month")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/figures/02_monthly_volume.png", bbox_inches="tight")
plt.show()

## 4. Word count distributions

In [ ]:
core_df = df[df["president"].isin(["Macri", "AF", "Milei"])]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, pres in zip(axes, ["Macri", "AF", "Milei"]):
    sub = core_df[core_df["president"] == pres]["n_words"]
    ax.hist(sub, bins=40, color=PRES_COLORS[pres], edgecolor="white", linewidth=0.4)
    ax.axvline(sub.median(), color="black", linestyle="--", linewidth=1.2,
               label=f"Median: {int(sub.median()):,}")
    ax.set_title(pres)
    ax.set_xlabel("Words per speech")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)

plt.suptitle("Word count distribution by president", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig("../outputs/figures/03_wordcount_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
data_to_plot = [core_df[core_df["president"] == p]["n_words"].values
                for p in ["Macri", "AF", "Milei"]]
bp = ax.boxplot(data_to_plot, patch_artist=True, notch=True,
                medianprops=dict(color="white", linewidth=2))
for patch, pres in zip(bp["boxes"], ["Macri", "AF", "Milei"]):
    patch.set_facecolor(PRES_COLORS[pres])
ax.set_xticklabels(["Macri", "AF", "Milei"])
ax.set_ylabel("Words per speech")
ax.set_title("Speech length comparison")
plt.tight_layout()
plt.savefig("../outputs/figures/04_wordcount_boxplot.png", bbox_inches="tight")
plt.show()

## 5. Monthly coverage — gaps relevant for BVAR

In [ ]:
all_months = pd.period_range(
    start="2015-12",
    end=df["date"].max().to_period("M"),
    freq="M"
)

month_counts = (
    df[df["date"].notna()]
    .assign(ym=lambda d: d["date"].dt.to_period("M"))
    .groupby("ym", observed=True)
    .size()
    .reindex(all_months, fill_value=0)
)

empty_months = month_counts[month_counts == 0]
print(f"Months with zero speeches: {len(empty_months)}")
print(empty_months.index.tolist())

In [ ]:
fig, ax = plt.subplots(figsize=(16, 3))
x = month_counts.index.to_timestamp()
y = month_counts.values

ax.fill_between(x, y, alpha=0.3, color="steelblue")
ax.plot(x, y, color="steelblue", linewidth=1)

for m in empty_months.index:
    ax.axvspan(m.to_timestamp(), (m+1).to_timestamp(),
               color="red", alpha=0.25)

for date, label in [("2015-12-10", "Macri"), ("2019-12-10", "AF"), ("2023-12-10", "Milei")]:
    ax.axvline(pd.Timestamp(date), color="black", linewidth=1, linestyle="--", alpha=0.5)

ax.set_title("Monthly speech count — red bands = months with no speeches (require interpolation in BVAR)")
ax.set_ylabel("Speeches")
plt.tight_layout()
plt.savefig("../outputs/figures/05_monthly_coverage.png", bbox_inches="tight")
plt.show()

## 6. Average speech length over time

In [ ]:
monthly_words = (
    df[df["date"].notna()]
    .assign(ym=lambda d: d["date"].dt.to_period("M"))
    .groupby("ym", observed=True)["n_words"]
    .mean()
    .reindex(all_months)
)

fig, ax = plt.subplots(figsize=(16, 3))
ax.plot(monthly_words.index.to_timestamp(), monthly_words.values,
        color="darkorange", linewidth=1.2)
for date, label in [("2015-12-10", "Macri"), ("2019-12-10", "AF"), ("2023-12-10", "Milei")]:
    ax.axvline(pd.Timestamp(date), color="black", linewidth=1, linestyle="--", alpha=0.5)
    ax.text(pd.Timestamp(date), ax.get_ylim()[1]*0.9, label,
            fontsize=8, ha="left", rotation=90, alpha=0.6)

ax.set_title("Average words per speech per month")
ax.set_ylabel("Mean words")
plt.tight_layout()
plt.savefig("../outputs/figures/06_avg_length_over_time.png", bbox_inches="tight")
plt.show()

## 7. Top speeches by length

In [ ]:
df.nlargest(10, "n_words")[["speech_id", "date", "president", "n_words", "url"]]

In [ ]:
df.nsmallest(10, "n_words")[["speech_id", "date", "president", "n_words", "url"]]

## 8. Summary

In [ ]:
print("=== CORPUS SUMMARY ===")
print(f"Total speeches : {len(df):,}")
print(f"Date range     : {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Empty months   : {len(empty_months)} (see Section 5)")
print()
print("Speeches per president:")
print(df["president"].value_counts().to_string())
print()
print("Word count stats:")
print(df.groupby("president", observed=True)["n_words"]
      .describe()[["mean","std","min","50%","max"]]
      .loc[[p for p in PRES_ORDER if p in df["president"].unique()]]
      .round(0).astype(int).to_string())